In [ ]:
!pip install -q MDAnalysis MDTraj torch-geometric

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#================================================================================
#SNIPPET 1: Model Initialization and Architecture
#================================================================================
#Description:
#    This module defines and initializes the Graph Attention Network (GAT) used
#    for classifying protein conformational states.
#
#Methodology:
#    1. Architecture: The model employs a two-layer GAT architecture (Velickovic et al.).
#       - Layer 1: 8 attention heads, concatenating outputs to capture diverse local neighborhoods.
#       - Layer 2: 8 attention heads, averaging outputs for the final node embedding.
#       - Readout: Global mean pooling aggregates node features into a graph-level vector.
#    2. Input/Output: Accepts node features (in_channels=3) and predicts binary class
#       probabilities (out_channels=2) via LogSoftmax.
#    3. State Loading: Loads pre-trained weights from a checkpoint to ensure
#       reproducibility of the reported inference results.

# ============================================================
# Imports
# ============================================================
import os
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

# ============================================================
# GPU / CPU selection
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# Paths (EDIT THESE ONLY)
# ============================================================
DATA_PATH  = "/content/drive/MyDrive/PATH/GAT_Two_Class_data.pt"
MODEL_PATH = "/content/drive/MyDrive/PATH/GAT_Two_Class_model.pt"

BASE_OUTPUT_DIR = "/content/drive/MyDrive/PATH/"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

# ============================================================
# Load data (PyTorch 2.6+ safe fix)
# ============================================================
print("Loading data_frames ...")
data_frames = torch.load(DATA_PATH, map_location="cpu", weights_only=False)
print("Total frames loaded:", len(data_frames))

# ============================================================
# Load checkpoint first, then infer architecture
# ============================================================
print("Loading model weights ...")
# ============================================================
# Model definition (must match training exactly)
# ============================================================
class GATNet(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # Matches Model_EVAL_two_class.py
        self.conv1 = GATConv(in_channels, 8, heads=8, concat=True)
        self.conv2 = GATConv(8 * 8, 16, heads=8, concat=False)
        self.lin = torch.nn.Linear(16, out_channels)

    def forward(self, x, edge_index):
        x, attn_weights_1 = self.conv1(x, edge_index, return_attention_weights=True)
        x = F.relu(x)
        x, attn_weights_2 = self.conv2(x, edge_index, return_attention_weights=True)

        # single-graph pooling (consistent with your training/eval)
        batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = global_mean_pool(x, batch)

        x = self.lin(x)
        out = F.log_softmax(x, dim=1)
        return out, attn_weights_1, attn_weights_2

# Instantiate and load weights
model = GATNet(in_channels=3, out_channels=2)

state = torch.load(MODEL_PATH, map_location=device, weights_only=False)

# Allow either a pure state_dict or a checkpoint dict containing state_dict
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]
elif isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]

model.load_state_dict(state, strict=True)
model = model.to(device)
model.eval()
print("✅ Model loaded successfully and ready on", device)

In [ ]:
#================================================================================
#SNIPPET 2: Performance Evaluation
#================================================================================
#Description:
#    Evaluates the model's predictive performance on a held-out test set to validate
#    generalizability before interpretability analysis.
#
#Methodology:
#    1. Test Set Segregation: A deterministic sampling strategy (every 10th frame)
#       is applied to disjoint sets of Label 0 and Label 1 data to create a
#       representative test split (N=~200 frames).
#    2. Inference: Forward pass is conducted in 'eval' mode (gradients disabled).
#    3. Metrics:
#       - Accuracy: Global ratio of correct predictions.
#       - Confusion Matrix: Normalized row-wise to percentage to visualize
#         Recall per class (5'->3' vs 3'->5'), controlling for any minor class imbalance.


import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import numpy as np

EVAL_DIR = os.path.join(BASE_OUTPUT_DIR, "EVAL")
os.makedirs(EVAL_DIR, exist_ok=True)

# --- Define test indices ---
# NOTE: this selects indices 1, 11, 21, ... (step=10)
test_indices_label0 = list(range(1, 1000, 10))
test_indices_label1 = list(range(1001, 2000, 10))
test_indices = sorted(test_indices_label0 + test_indices_label1)

# --- Build train/test splits using ORIGINAL indexing ---
test_frames  = [data_frames[i] for i in test_indices]
train_orig_indices = [i for i in range(len(data_frames)) if i not in set(test_indices)]
train_frames = [data_frames[i] for i in train_orig_indices]

# --- Sanity: class counts in test ---
test_label_0_count = sum(int(d.y.item() == 0) for d in test_frames)
test_label_1_count = sum(int(d.y.item() == 1) for d in test_frames)
print(f"Test set size: Label 0: {test_label_0_count}, Label 1: {test_label_1_count}")
print(f"Train size: {len(train_frames)} | Test size: {len(test_frames)}")

# --- Evaluate on test set ---
y_true, y_pred = [], []
model.eval()
with torch.no_grad():
    for d in test_frames:
        d = d.to(device)
        logits, _, _ = model(d.x, d.edge_index)
        y_true.append(int(d.y.item()))
        y_pred.append(int(logits.argmax(dim=1).item()))

test_acc = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

# --- Confusion matrix  ---
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

cm_percent = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(6, 5))
ax = sns.heatmap(
    cm_percent,
    cmap="Blues",
    cbar=True,
    cbar_kws={"label": "Percentage (%)"},
    xticklabels=["5'→3'", "3'→5'"],
    yticklabels=["5'→3'", "3'→5'"]
)

ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")
ax.set_title("Confusion Matrix (Test Set)")

cm_path = os.path.join(EVAL_DIR, "confusion_matrix.png")
plt.tight_layout()
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Confusion matrix saved at: {cm_path}")

In [ ]:
#================================================================================
#SNIPPET 3: Data Stratification for Interpretability
#================================================================================
#Description:
#    Prepares the dataset for attention extraction by isolating training frames
#    into class-specific clusters.
#
#Methodology:
#    1. Subset Selection: Focuses analysis on the training distribution to map
#       learned features.
#    2. Stratification: Frames are dictionary-indexed by their ground-truth label
#       (Label 1 vs. Label 2). This allows for differential attention analysis:
#       comparing how the network 'views' the 5'->3' conformation versus the
#       3'->5' conformation.

# ============================================================
# SWITCH ANALYSIS TO TRAIN SET ONLY
#   - data_frames becomes train_frames
#   - label_ranges_mon12 is rebuilt to refer to TRAIN indices
# ============================================================
orig_to_train = {orig_i: new_i for new_i, orig_i in enumerate(train_orig_indices)}

# ============================================================
# Switch analysis to TRAIN SET ONLY
#   - data_frames becomes train_frames
#   - label_ranges_mon12 is rebuilt to match NEW indexing (0..len(train_frames)-1)
# ============================================================
data_frames = train_frames

label_ranges_mon12 = {
    "label_1": [i for i, d in enumerate(data_frames) if int(d.y.item()) == 0],
    "label_2": [i for i, d in enumerate(data_frames) if int(d.y.item()) == 1],
}

print("Train-only analysis frames:")
print("  label_1:", len(label_ranges_mon12["label_1"]))
print("  label_2:", len(label_ranges_mon12["label_2"]))

In [ ]:
#================================================================================
#SNIPPET 4: Biological Mapping and Statistical Utilities
#================================================================================
#Description:
#    Provides the infrastructure to translate abstract graph indices into biological
#    residues and compute statistical properties of the attention mechanism.
#
#Methodology:
#    1. Residue Mapping: Maps graph node indices (0-N) to physical residue numbers
#       and functional domains (e.g., 'Walker A', 'ISM') based on the specific
#       protein sequence.
#    2. Attention Aggregation (get_attention_vector): Sums attention coefficients
#       across the 8 GAT heads to produce a single scalar weight per edge.
#    3. Statistical Filtering (filter_and_stats):
#       - Calculates 'Presence': The % of frames an edge exists in the trajectory.
#       - Filters noise: Removes edges below a defined presence threshold.
#       - Computes Stability: Calculates Mean, Std Dev, and Standard Error (SE)
#         of attention weights to identify stable vs. transient interactions.

import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict, OrderedDict
from matplotlib.colors import Normalize


# ------------------------
# Node → residue mapping
# ------------------------
MON1_NODE_MIN, MON1_NODE_MAX = 0, 275
MON2_NODE_MIN, MON2_NODE_MAX = 276, 587
mon1_offset = 35
mon2_offset = 37

# ------------------------
# Sequence & residue label
# ------------------------
sequence = (
    "MTEAQAIAKQLGGVKPDDEWLQAEIARLKGKSIVPLQQVKTLXDWLDGKRKARKSCRVVGESRTGKTVACDAYRYRXKPQQEAGRPPTVPVVYIRPHQKCGPKDL"
    "FKKITEYLKYRVTKGTVSDFRDRTIEVLKGCGVEMLIIDEADRLKPETFADVRDIAEDLGIAVVLVGTDRLDAVIKRDEQVLERFRAXLRFGKLSGEDFKNTVEM"
    "WEQMVLKLPVSSNLKSKEMLRILTSATEGYIGRLDEILREAAIRSLSRGLKKIDKAVLQEVAKEYK"
)

def get_residue_label(res):
    if res < 313:
        idx = res - 35
        num = res - 34
    else:
        idx = res - 313
        num = res - 312
    return f"{sequence[idx]}{num}" if 0 <= idx < len(sequence) else f"X{res}"

# ------------------------
# Regions (same as your script)
# ------------------------
monomer1_regions = {
    "NTER alpha helix": range(35, 93),
    "Walker A": range(94, 101),
    "Walker_A_ISM Linker": range(102, 128),
    "ISM": range(129, 173),
    "Walker B": range(174, 179),
    "Linker walker B C TER": range(180, 230),
    "CTER alpha helix": range(231, 310),
}

monomer2_regions = {
    "NTER alpha helix": range(313, 371),
    "Walker A": range(372, 379),
    "Walker_A_ISM Linker": range(380, 406),
    "ISM": range(407, 451),
    "Walker B": range(452, 457),
    "Linker walker B C TER": range(458, 508),
    "CTER alpha helix": range(509, 588),
}

# ------------------------
# Attention extractor (layer-aware)
# ------------------------
def get_attention_vector(alpha1, alpha2, mode="sum"):
    """
    Returns one scalar per edge (length E) for layer1, layer2, or sum.

    NOTE (PyG GATConv): when return_attention_weights=True, attention is returned as:
        (edge_index_used, alpha_tensor)
    This function accepts either the raw alpha tensor OR that tuple.
    """
 # Unpack PyG tuple format if needed
    if isinstance(alpha1, (tuple, list)):
        alpha1 = alpha1[1]
    if isinstance(alpha2, (tuple, list)):
        alpha2 = alpha2[1]
# Collapse multi-dim (e.g., heads) -> one scalar per edge
    a1 = alpha1 if alpha1.dim() == 1 else alpha1.sum(dim=1)
    a2 = alpha2 if alpha2.dim() == 1 else alpha2.sum(dim=1)

    if mode == "layer1":
        att = a1
    elif mode == "layer2":
        att = a2
    elif mode == "sum":
        att = a1 + a2
    else:
        raise ValueError("mode must be 'layer1', 'layer2', or 'sum'")

    return att.detach().cpu().numpy()

def filter_and_stats(edge_vals, nframes, pct):
    out = {}
    for e, v in edge_vals.items():
        pres = 100 * len(v) / nframes
        if pres > pct:
            v = np.array(v, dtype=float)
            out[e] = (float(v.mean()), float(v.std()), float(v.std()/np.sqrt(len(v))), float(pres))
    return out

def save_stats(filtered, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write("Source Target Avg Std SE Presence\n")
        for (s, t), (a, b, c, p) in filtered.items():
            f.write(f"{s} {t} {a:.4f} {b:.4f} {c:.4f} {p:.2f}\n")

In [ ]:
# ============================================================
# SNIPPET 5: Inter-Monomer Interaction Analysis (BIDIRECTIONAL)
# ==========================================================
# Description:
#    Quantifies the structural communication between Monomer 1 and Monomer 2 by
#    extracting and aggregating bidirectional attention weights.
#
# Methodology:
#    1. Edge Extraction: Iterates through all frames, identifying edges that span
#       across the Mon1-Mon2 interface (Bidirectional: Mon1->Mon2 and Mon2->Mon1).
#    2. Dynamics (Variance): Computes the variance (std^2) of attention weights
#       over time. High variance implies a flexible/dynamic interface; low variance
#       implies a rigid contact.
#    3. Coarse-Graining: Aggregates residue-level attention into Region-Level matrices
#       (e.g., ISM -> Walker A) to visualize domain-domain communication.
#    4. Visualization: Generates heatmaps (Sum & Variance) and histograms to
#       compare interaction fingerprints between the two classes.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, OrderedDict

# ------------------------
# Device (GPU if available)
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print("Snippet 5 running on:", device)

# =========================
# USER CONTROL (single place)
# =========================
PRESENCE_THRESHOLD_INTER = 10.0   # <-- edit this
YMAX_COMBINED_HIST_INTER = 6.5    # <-- edit this if needed

# =========================
# Helper: save presence
# =========================
def save_presence_txt(edge_pct, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write("Source Target PresencePercent\n")
        for (s, t), p in sorted(edge_pct.items()):
            f.write(f"{s} {t} {p:.2f}\n")

# =========================
# Presence histogram
# =========================
def plot_presence_hist(edge_pct, title, out_png):
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    if len(edge_pct) == 0:
        print("WARNING: no edges for presence histogram:", out_png)
        return
    vals = list(edge_pct.values())
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(vals)), vals, color="skyblue")
    plt.xlabel("Edges")
    plt.ylabel("Presence (%)")
    plt.title(title)
    plt.xticks([])
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

# =========================
# Avg-attention scatter (two versions)
# =========================
def plot_inter_scatter(filtered_l1, filtered_l2, out_png, adjusted_ticks=False, title_prefix=""):
    def unpack(f):
        xs = [e[0] for e in f.keys()]
        ys = [e[1] for e in f.keys()]
        cs = [v[0] for v in f.values()]  # avg
        return xs, ys, cs

    x1, y1, c1 = unpack(filtered_l1)
    x2, y2, c2 = unpack(filtered_l2)


    if len(c1) == 0 or len(c2) == 0:
        print("WARNING: empty scatter:", out_png)
        return

    vmin = min(min(c1), min(c2))
    vmax = max(max(c1), max(c2))

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # ---- LEFT: label_1 ----
    ax1 = axes[0]
    sc1 = ax1.scatter(x1, y1, c=c1, cmap="viridis", vmin=vmin, vmax=vmax, s=10)
    ax1.set_title(f"{title_prefix}Avg Attention | 5'→3'")
    ax1.set_xlabel("Residue (Bound)")
    ax1.set_ylabel("Residue (Incoming)")
    plt.colorbar(sc1, ax=ax1, label="Average Attention")

    # ---- RIGHT: label_2 ----
    ax2 = axes[1]
    sc2 = ax2.scatter(x2, y2, c=c2, cmap="viridis", vmin=vmin, vmax=vmax, s=10)
    ax2.set_title(f"{title_prefix}Avg Attention | 3'→5'")
    ax2.set_xlabel("Residue (Bound)")
    ax2.set_ylabel("Residue (Incoming)")
    plt.colorbar(sc2, ax=ax2, label="Average Attention")

    # ---- adjusted ticks (apply to the real axes only) ----
    if adjusted_ticks:
        xticks = np.arange(35, 311, 25)
        yticks = np.arange(313, 589, 25)
        xlabels = np.arange(1, 277, 25)
        ylabels = np.arange(1, 277, 25)
        for ax in (ax1, ax2):
            ax.set_xticks(xticks)
            ax.set_xticklabels(xlabels)
            ax.set_yticks(yticks)
            ax.set_yticklabels(ylabels)


    plt.tight_layout()
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    plt.savefig(out_png, dpi=300)
    plt.close()

# =========================
# Variance scatter (variance = std^2), shared color scale
# =========================
def extract_variance_scatter_data(filtered_edges):
    sources, targets, variances = [], [], []
    for edge, stats in filtered_edges.items():
        std_dev = stats[1]
        variance = std_dev**2
        sources.append(edge[0])
        targets.append(edge[1])
        variances.append(float(variance))
    return sources, targets, variances

def plot_attention_variance_scatter(filtered_edges_label1, filtered_edges_label2, out_png, title_prefix=""):
    plt.figure(figsize=(15, 5))

    s1, t1, v1 = extract_variance_scatter_data(filtered_edges_label1)
    s2, t2, v2 = extract_variance_scatter_data(filtered_edges_label2)


    if len(v1) == 0 or len(v2) == 0:
        print("WARNING: empty variance scatter:", out_png)
        return

    allv = v1 + v2
    vmin = min(allv)
    vmax = max(allv) if max(allv) > vmin else (vmin + 1e-12)

    plt.subplot(1, 2, 1)
    sc1 = plt.scatter(s1, t1, c=v1, cmap="viridis", vmin=vmin, vmax=vmax, s=12)
    plt.colorbar(sc1, label="Variance (Std^2)")
    plt.title(f"{title_prefix}Variance Scatter | 5'→3'")
    plt.xlabel("Residue (Bound)")
    plt.ylabel("Residue (Incoming)")

    plt.subplot(1, 2, 2)
    sc2 = plt.scatter(s2, t2, c=v2, cmap="viridis", vmin=vmin, vmax=vmax, s=12)
    plt.colorbar(sc2, label="Variance (Std^2)")
    plt.title(f"{title_prefix}Variance Scatter | 3'→5'")
    plt.xlabel("Residue (Bound)")
    plt.ylabel("Residue (Incoming)")

    plt.tight_layout()
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    plt.savefig(out_png, dpi=300)
    plt.close()

# =========================
# Region SUM + TIME-VARIANCE (from edge std^2)
# =========================
def region_sum_and_time_variance_from_filtered(filtered_edges, row_regions, col_regions, agg="sum"):
    buckets_avg = defaultdict(list)
    buckets_var = defaultdict(list)

    for (s, t), (avg, std, se, pres) in filtered_edges.items():
        rn = None
        cn = None

        for rname, rr in row_regions.items():
            if s in rr:
                rn = rname
                break
        for cname, cr in col_regions.items():
            if t in cr:
                cn = cname
                break

        if rn is None or cn is None:
            continue

        buckets_avg[(rn, cn)].append(float(avg))
        buckets_var[(rn, cn)].append(float(std) ** 2)

    region_sum_avg = pd.DataFrame(0.0, index=list(row_regions.keys()), columns=list(col_regions.keys()))
    region_var_time = pd.DataFrame(0.0, index=list(row_regions.keys()), columns=list(col_regions.keys()))

    for (rn, cn), avgs in buckets_avg.items():
        region_sum_avg.loc[rn, cn] = float(np.sum(avgs))

    for (rn, cn), vars_ in buckets_var.items():
        if len(vars_) == 0:
            region_var_time.loc[rn, cn] = 0.0
        else:
            if agg == "sum":
                region_var_time.loc[rn, cn] = float(np.sum(vars_))
            elif agg == "mean":
                region_var_time.loc[rn, cn] = float(np.mean(vars_))
            else:
                raise ValueError("agg must be 'sum' or 'mean'")

    return region_sum_avg, region_var_time

# =========================
# Region plotting: print values only if non-zero
# =========================
def plot_two_panel_heatmaps_with_values_nonzero(mat1, mat2, out_png, title1, title2, cmap="Spectral_r", fmt="{:.2f}", eps=0.0):
    fig, axes = plt.subplots(1, 2, figsize=(20, 9))
    mats = [mat1, mat2]
    titles = [title1, title2]

    for ax, mat, title in zip(axes, mats, titles):
        vmax = mat.values.max() if mat.values.max() > 0 else 1.0
        im = ax.imshow(mat.values, cmap=cmap, vmin=0, vmax=vmax, aspect="auto")
        ax.set_title(title)

        ax.set_xticks(range(len(mat.columns)))
        ax.set_xticklabels(mat.columns, rotation=45, ha="right")
        ax.set_yticks(range(len(mat.index)))
        ax.set_yticklabels(mat.index)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = float(mat.iat[i, j])
                if abs(val) > eps:
                    ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=8)

        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    plt.savefig(out_png, dpi=300)
    plt.close()

# =========================
# write combined histogram edge lists by region-pair (TXT)
# =========================
def _find_region_name(res, regions_dict):
    for rname, rr in regions_dict.items():
        if res in rr:
            return rname
    return None

def write_edges_by_regionpair_txt(filtered_edges, row_regions, col_regions, out_txt, modified=True):
    """
    Writes edges grouped by region-pair.
    filtered_edges: dict edge -> (avg, std, se, pres)
    """
    os.makedirs(os.path.dirname(out_txt), exist_ok=True)

    # bucket edges by region-pair
    buckets = defaultdict(list)
    for (s, t), (avg, std, se, pres) in filtered_edges.items():
        rn = _find_region_name(s, row_regions)
        cn = _find_region_name(t, col_regions)
        if rn is None or cn is None:
            continue
        buckets[(rn, cn)].append((s, t, avg, std, se, pres))

    # sort region-pairs by name for stable output
    keys_sorted = sorted(buckets.keys(), key=lambda x: (x[0], x[1]))

    with open(out_txt, "w") as f:
        f.write("# RegionPair  Source  Target  EdgeLabel  Avg  Std  SE  Presence\n")
        for (rn, cn) in keys_sorted:
            f.write(f"\n## {rn} -> {cn}\n")
            # sort edges within region-pair by Avg desc (then Presence desc)
            edges = sorted(buckets[(rn, cn)], key=lambda z: (-z[2], -z[5], z[0], z[1]))
            for s, t, avg, std, se, pres in edges:
                if modified:
                    edge_label = f"{get_residue_label(int(s))}-{get_residue_label(int(t))}"
                else:
                    edge_label = f"{int(s)}-{int(t)}"
                f.write(f"{rn}->{cn}  {int(s)}  {int(t)}  {edge_label}  {avg:.6f}  {std:.6f}  {se:.6f}  {pres:.2f}\n")

# =========================
# Combined histograms (modified/unmodified) + SAME sorting logic
# =========================
def df_from_filtered(filtered_edges):
    rows = []
    for (s, t), (avg, std, se, pres) in filtered_edges.items():
        rows.append((int(s), int(t), float(avg), float(se)))
    return pd.DataFrame(rows, columns=["Source", "Target", "Average_Attention", "Std_Error"])

def plot_combined_histograms(df_attention, region_rows, region_cols, out_dir, label_text, ylim_max=4.5):
    os.makedirs(out_dir, exist_ok=True)
    plt.figure(figsize=(20, 12))
    color_map = plt.cm.get_cmap("tab20", len(region_rows) * len(region_cols))

    edge_labels_mod = []
    edge_labels_unmod = []
    avg_attention = []
    std_errors = []
    colors = []
    legend_handles = []
    region_color_mapping = OrderedDict()

    color_idx = 0
    for r1 in region_rows.keys():
        for r2 in region_cols.keys():
            region_color_mapping[f"{r1} → {r2}"] = color_map(color_idx)
            color_idx += 1

    for r1_name, r1_range in region_rows.items():
        for r2_name, r2_range in region_cols.items():
            sub = df_attention[(df_attention["Source"].isin(r1_range)) & (df_attention["Target"].isin(r2_range))].copy()
            if sub.empty:
                continue

            legend_label = f"{r1_name} → {r2_name}"
            color = region_color_mapping[legend_label]
            if legend_label not in [h.get_label() for h in legend_handles]:
                legend_handles.append(plt.Line2D([0], [0], color=color, lw=4, label=legend_label))

            for _, row in sub.iterrows():
                s = int(row["Source"])
                t = int(row["Target"])
                edge_labels_mod.append(f"{get_residue_label(s)}-{get_residue_label(t)}")
                edge_labels_unmod.append(f"{s}-{t}")
                avg_attention.append(float(row["Average_Attention"]))
                std_errors.append(float(row["Std_Error"]))
                colors.append(color)

    if len(edge_labels_mod) == 0:
        print("No edges for combined histogram:", label_text)
        plt.close()
        return

    # Modified
    plt.bar(range(len(avg_attention)), avg_attention, yerr=std_errors, color=colors, capsize=5, alpha=0.85, width=0.6)
    plt.xticks(range(len(avg_attention)), edge_labels_mod, rotation=90, fontsize=7)
    plt.ylabel("Average Attention")
    plt.ylim(0, ylim_max)
    plt.title(f"Combined Histogram (Modified Labels) | {label_text}")
    plt.legend(handles=legend_handles, title="Region–Region Interactions", loc="upper right", fontsize=9, title_fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"combined_histogram_modified_{label_text}.png"), dpi=300)
    plt.close()

    # Unmodified
    plt.figure(figsize=(20, 12))
    plt.bar(range(len(avg_attention)), avg_attention, yerr=std_errors, color=colors, capsize=5, alpha=0.85, width=0.6)
    plt.xticks(range(len(avg_attention)), edge_labels_unmod, rotation=90, fontsize=7)
    plt.ylabel("Average Attention")
    plt.ylim(0, ylim_max)
    plt.title(f"Combined Histogram (Unmodified Labels) | {label_text}")
    plt.legend(handles=legend_handles, title="Region–Region Interactions", loc="upper right", fontsize=9, title_fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"combined_histogram_unmodified_{label_text}.png"), dpi=300)
    plt.close()

# ============================================================
# MAIN RUN (layer1 / layer2 / sum)
# ============================================================
for mode in ["layer1", "layer2", "sum"]:
    print("\n=== INTER mon1–mon2 (BIDIR) mode:", mode, "===")

    base_mode = os.path.join(BASE_OUTPUT_DIR, f"INTER_BIDIR_{mode}")
    out_avg  = os.path.join(base_mode, "mon1_mon2_avg_attention")
    out_hist = os.path.join(base_mode, "mon1_mon2_presence_histograms")
    out_reg  = os.path.join(base_mode, "region_attention_analysis")
    out_comb = os.path.join(base_mode, "mon1_mon2_region_histograms")

    for d in [out_avg, out_hist, out_reg, out_comb]:
        os.makedirs(d, exist_ok=True)

    filtered = {}  # label -> filtered edge stats

    # ---------- Per label: presence + avg stats ----------
    for lab, frange in label_ranges_mon12.items():
        edge_vals = defaultdict(list)
        edge_counter = defaultdict(int)

        for fi in frange:
            df = data_frames[fi].to(device)
            with torch.no_grad():
                _, alpha1, alpha2 = model(df.x, df.edge_index)

            att = get_attention_vector(alpha1, alpha2, mode=mode)  # from snippet 2
            ei = (alpha1[0] if isinstance(alpha1, (tuple, list)) else df.edge_index).detach().cpu().numpy()

            # ---------- BIDIRECTIONAL EDGE COLLECTION ----------
            for i, (s, t) in enumerate(zip(ei[0], ei[1])):
                # forward: mon1 -> mon2
                if (MON1_NODE_MIN <= s <= MON1_NODE_MAX) and (MON2_NODE_MIN <= t <= MON2_NODE_MAX):
                    edge = (int(s + mon1_offset), int(t + mon2_offset))
                    edge_counter[edge] += 1
                    edge_vals[edge].append(float(att[i]))

                # backward: mon2 -> mon1  (map to same key by swapping)
                elif (MON2_NODE_MIN <= s <= MON2_NODE_MAX) and (MON1_NODE_MIN <= t <= MON1_NODE_MAX):
                    edge = (int(t + mon1_offset), int(s + mon2_offset))
                    edge_counter[edge] += 1
                    edge_vals[edge].append(float(att[i]))
            # -----------------------------------------------

        # presence (%)
        edge_pct = {e: (c / len(frange)) * 100.0 for e, c in edge_counter.items()}
        save_presence_txt(edge_pct, os.path.join(out_hist, f"mon1_mon2_presence_{lab}.txt"))
        plot_presence_hist(
            edge_pct,
            title=f"Mon1–Mon2 (BIDIR) Presence (%) | {lab} | {mode}",
            out_png=os.path.join(out_hist, f"mon1_mon2_presence_{lab}.png")
        )

        # avg attention stats after filtering
        filt = filter_and_stats(edge_vals, len(frange), PRESENCE_THRESHOLD_INTER)
        filtered[lab] = filt
        save_stats(filt, os.path.join(out_avg, f"mon1_mon2_avg_attention_{lab}.txt"))

    # ---------- Scatter (avg attention) ----------
    plot_inter_scatter(
        filtered["label_1"], filtered["label_2"],
        out_png=os.path.join(out_avg, "mon1_mon2_scatter.png"),
        adjusted_ticks=False,
        title_prefix=f"[{mode}] "
    )
    plot_inter_scatter(
        filtered["label_1"], filtered["label_2"],
        out_png=os.path.join(out_avg, "mon1_mon2_scatter_adjusted_labels.png"),
        adjusted_ticks=True,
        title_prefix=f"[{mode}] "
    )

    # ---------- Variance scatter (std^2) ----------
    plot_attention_variance_scatter(
        filtered["label_1"], filtered["label_2"],
        out_png=os.path.join(out_avg, f"mon1_mon2_variance_scatter_{mode}.png"),
        title_prefix=f"[{mode}] "
    )

    # ---------- Region SUM + TIME-VARIANCE (std^2 aggregated), heatmaps + CSV ----------
    sum1, var_time1 = region_sum_and_time_variance_from_filtered(
        filtered["label_1"], monomer1_regions, monomer2_regions, agg="sum"
    )
    sum2, var_time2 = region_sum_and_time_variance_from_filtered(
        filtered["label_2"], monomer1_regions, monomer2_regions, agg="sum"
    )

    # Save CSVs
    sum1.to_csv(os.path.join(out_reg, "region_sum_label_1.csv"))
    sum2.to_csv(os.path.join(out_reg, "region_sum_label_2.csv"))
    var_time1.to_csv(os.path.join(out_reg, "region_time_variance_sum_label_1.csv"))
    var_time2.to_csv(os.path.join(out_reg, "region_time_variance_sum_label_2.csv"))

    # Plot (use *_plot)
    plot_two_panel_heatmaps_with_values_nonzero(
        sum1, sum2,
        out_png=os.path.join(out_reg, "region_attention_heatmap_SUM.png"),
        title1=f"Region SUM | 5′→3′ | {mode}",
        title2=f"Region SUM | 3′→5′ | {mode}",
        fmt="{:.2f}",
        eps=0.0
    )

    plot_two_panel_heatmaps_with_values_nonzero(
        var_time1, var_time2,
        out_png=os.path.join(out_reg, "region_attention_heatmap_TIME_VARIANCE_SUM.png"),
        title1=f"Region TIME-VARIANCE | 5′→3′ | {mode}",
        title2=f"Region TIME-VARIANCE | 3′→5′ | {mode}",
        fmt="{:.2e}",
        eps=0.0
    )

    # ---------- Combined histograms (modified + unmodified) ----------
    df1 = df_from_filtered(filtered["label_1"])
    df2 = df_from_filtered(filtered["label_2"])

    plot_combined_histograms(df1, monomer1_regions, monomer2_regions, out_comb, f"label_1_{mode}", ylim_max=YMAX_COMBINED_HIST_INTER)
    plot_combined_histograms(df2, monomer1_regions, monomer2_regions, out_comb, f"label_2_{mode}", ylim_max=YMAX_COMBINED_HIST_INTER)

    # ---------- NEW: Text outputs for combined histogram edge lists grouped by region-pair ----------
    txt_mod_1 = os.path.join(out_comb, f"combined_edges_by_regionpair_modified_label_1_{mode}.txt")
    txt_unm_1 = os.path.join(out_comb, f"combined_edges_by_regionpair_unmodified_label_1_{mode}.txt")
    txt_mod_2 = os.path.join(out_comb, f"combined_edges_by_regionpair_modified_label_2_{mode}.txt")
    txt_unm_2 = os.path.join(out_comb, f"combined_edges_by_regionpair_unmodified_label_2_{mode}.txt")

    write_edges_by_regionpair_txt(filtered["label_1"], monomer1_regions, monomer2_regions, txt_mod_1, modified=True)
    write_edges_by_regionpair_txt(filtered["label_1"], monomer1_regions, monomer2_regions, txt_unm_1, modified=False)
    write_edges_by_regionpair_txt(filtered["label_2"], monomer1_regions, monomer2_regions, txt_mod_2, modified=True)
    write_edges_by_regionpair_txt(filtered["label_2"], monomer1_regions, monomer2_regions, txt_unm_2, modified=False)

print("\nDONE: Snippet 5 complete (INTER mon1–mon2, BIDIR) for layer1/layer2/sum.")

In [ ]:
# ============================================================
# SNIPPET 6: Correlation between layer1, layer2, and sum
# =======================================================
# Description:
#    Investigates the hierarchical consistency of learned features by correlating
#    attention maps between GAT layer1, layer2, and sum.
#
# Methodology:
#    1. Pearson Correlation (r): Assesses linear scaling of attention weights
#       between layers.
#    2. Spearman Rank Correlation (rho): Assesses the preservation of edge
#       ranking (importance order), accounting for non-linear transformations (ReLU).
#    3. Statistical Significance: Computes p-values for both metrics to quantify
#       the reliability of the observed correlations.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# ------------------------
# SET THIS ONLY
# ------------------------
INTER_VARIANT = "INTER_BIDIR"

CORR_OUT = os.path.join(BASE_OUTPUT_DIR, f"CORR_{INTER_VARIANT}")
os.makedirs(CORR_OUT, exist_ok=True)

# ------------------------
# Helpers
# ------------------------
def read_edge_stats_txt(path):
    """
    Reads saved edge stats txt:
      Source Target Avg Std SE Presence
    Returns DataFrame with Source, Target as int and Avg/Std/SE/Presence float.
    """
    rows = []
    with open(path, "r") as f:
        header = f.readline().strip().split()
        if len(header) < 6 or header[:2] != ["Source", "Target"]:
            raise ValueError(f"Unexpected header in {path}: {header}")
        for line in f:
            if not line.strip():
                continue
            s, t, avg, std, se, pres = line.split()
            rows.append((int(s), int(t), float(avg), float(std), float(se), float(pres)))
    return pd.DataFrame(rows, columns=["Source", "Target", "Avg", "Std", "SE", "Presence"])

def edge_merge_for_corr(df_a, df_b, name_a, name_b):
    return df_a.merge(df_b, on=["Source", "Target"], how="inner",
                      suffixes=(f"_{name_a}", f"_{name_b}"))

def pearson_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan
    r, p = pearsonr(x, y)
    return float(r), float(p)

def spearman_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan
    r, p = spearmanr(x, y)
    return float(r), float(p)

def scatter_with_corr(x, y, title, xlabel, ylabel, out_png):
    pr, pp = pearson_stats(x, y)
    sr, sp = spearman_stats(x, y)
    plt.figure(figsize=(6, 6))
    plt.scatter(x, y, s=14, alpha=0.7)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(
        f"{title}\n"
        f"Pearson r={pr:.3f} (p={pp:.2e}) | Spearman ρ={sr:.3f} (p={sp:.2e}) | N={len(x)}"
    )
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()
    return pr, pp, sr, sp

def load_region_csv(path):
    return pd.read_csv(path, index_col=0)

def region_corr_stats(dfA, dfB):
    # flatten in same order
    A = dfA.values.flatten().astype(float)
    B = dfB.values.flatten().astype(float)
    pr, pp = pearson_stats(A, B)
    sr, sp = spearman_stats(A, B)
    return pr, pp, sr, sp, int(len(A))

def heatmap_corr_matrix(pearson_vals_dict, title, out_png):
    """
    pearson_vals_dict: dict like {("layer1","layer2"): pearson_r, ...}
    Makes a 3x3 heatmap (Pearson only) for quick view.
    """
    labels = ["layer1", "layer2", "sum"]
    mat = np.full((3, 3), np.nan, dtype=float)
    for i, a in enumerate(labels):
        for j, b in enumerate(labels):
            if a == b:
                mat[i, j] = 1.0
            elif (a, b) in pearson_vals_dict:
                mat[i, j] = pearson_vals_dict[(a, b)]
            elif (b, a) in pearson_vals_dict:
                mat[i, j] = pearson_vals_dict[(b, a)]

    finite = mat[np.isfinite(mat)]
    vmax = np.max(np.abs(finite)) if finite.size > 0 else 1.0

    plt.figure(figsize=(6, 5))
    im = plt.imshow(mat, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    plt.xticks(range(3), labels)
    plt.yticks(range(3), labels)
    for i in range(3):
        for j in range(3):
            if np.isfinite(mat[i, j]):
                plt.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center", fontsize=10)
    plt.title(title + " (Pearson r)")
    plt.colorbar(im, label="r")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

# ------------------------
# Paths
# ------------------------
def mode_base(mode):
    return os.path.join(BASE_OUTPUT_DIR, f"{INTER_VARIANT}_{mode}")

def edge_stats_path(mode, label):
    return os.path.join(mode_base(mode), "mon1_mon2_avg_attention",
                        f"mon1_mon2_avg_attention_{label}.txt")

def region_sum_path(mode, label):
    return os.path.join(mode_base(mode), "region_attention_analysis",
                        f"region_sum_{label}.csv")

def region_timevar_path(mode, label):
    return os.path.join(mode_base(mode), "region_attention_analysis",
                        f"region_time_variance_sum_{label}.csv")

# ============================================================
# REGION-LEVEL correlations (Region SUM maps)
# ============================================================
for label in ["label_1", "label_2"]:
    rsum_l1  = load_region_csv(region_sum_path("layer1", label))
    rsum_l2  = load_region_csv(region_sum_path("layer2", label))
    rsum_sum = load_region_csv(region_sum_path("sum", label))

    # Ensure identical index/columns order
    rsum_l2  = rsum_l2.reindex(index=rsum_l1.index, columns=rsum_l1.columns)
    rsum_sum = rsum_sum.reindex(index=rsum_l1.index, columns=rsum_l1.columns)

    pairs = {}
    pairs[("layer1", "layer2")] = region_corr_stats(rsum_l1, rsum_l2)  # pr,pp,sr,sp,N
    pairs[("layer1", "sum")]    = region_corr_stats(rsum_l1, rsum_sum)
    pairs[("layer2", "sum")]    = region_corr_stats(rsum_l2, rsum_sum)

    rows = []
    pearson_vals = {}
    for (a, b), (pr, pp, sr, sp, N) in pairs.items():
        rows.append((f"{a}_vs_{b}", N, pr, pp, sr, sp))
        pearson_vals[(a, b)] = pr

    out = pd.DataFrame(rows, columns=[
        "Comparison", "N_cells",
        "Pearson_r", "Pearson_p",
        "Spearman_rho", "Spearman_p"
    ])
    out.to_csv(os.path.join(CORR_OUT, f"region_sum_corr_{label}.csv"), index=False)
    with open(os.path.join(CORR_OUT, f"region_sum_corr_{label}.txt"), "w") as f:
        f.write(out.to_string(index=False) + "\n")

    # Scatter plots (flatten)
    def _plot_flat(A, B, name):
        x = A.values.flatten()
        y = B.values.flatten()
        scatter_with_corr(
            x, y,
            title=f"Region SUM Corr ({INTER_VARIANT}) ({label}) {name}",
            xlabel=name.split("_vs_")[0], ylabel=name.split("_vs_")[1],
            out_png=os.path.join(CORR_OUT, f"region_sum_scatter_{label}_{name}.png")
        )

    _plot_flat(rsum_l1, rsum_l2, "layer1_vs_layer2")
    _plot_flat(rsum_l1, rsum_sum, "layer1_vs_sum")
    _plot_flat(rsum_l2, rsum_sum, "layer2_vs_sum")

    heatmap_corr_matrix(
        pearson_vals,
        f"REGION SUM Corr ({INTER_VARIANT}) ({label})",
        os.path.join(CORR_OUT, f"region_sum_corr_heatmap_{label}.png")
    )

# ============================================================
# REGION-LEVEL correlations (Region TIME-VARIANCE maps)
# ============================================================
for label in ["label_1", "label_2"]:
    rtv_l1  = load_region_csv(region_timevar_path("layer1", label))
    rtv_l2  = load_region_csv(region_timevar_path("layer2", label))
    rtv_sum = load_region_csv(region_timevar_path("sum", label))

    rtv_l2  = rtv_l2.reindex(index=rtv_l1.index, columns=rtv_l1.columns)
    rtv_sum = rtv_sum.reindex(index=rtv_l1.index, columns=rtv_l1.columns)

    pairs = {}
    pairs[("layer1", "layer2")] = region_corr_stats(rtv_l1, rtv_l2)
    pairs[("layer1", "sum")]    = region_corr_stats(rtv_l1, rtv_sum)
    pairs[("layer2", "sum")]    = region_corr_stats(rtv_l2, rtv_sum)

    rows = []
    pearson_vals = {}
    for (a, b), (pr, pp, sr, sp, N) in pairs.items():
        rows.append((f"{a}_vs_{b}", N, pr, pp, sr, sp))
        pearson_vals[(a, b)] = pr

    out = pd.DataFrame(rows, columns=[
        "Comparison", "N_cells",
        "Pearson_r", "Pearson_p",
        "Spearman_rho", "Spearman_p"
    ])
    out.to_csv(os.path.join(CORR_OUT, f"region_timevar_corr_{label}.csv"), index=False)
    with open(os.path.join(CORR_OUT, f"region_timevar_corr_{label}.txt"), "w") as f:
        f.write(out.to_string(index=False) + "\n")

    def _plot_flat(A, B, name):
        x = A.values.flatten()
        y = B.values.flatten()
        scatter_with_corr(
            x, y,
            title=f"Region TIME-VAR Corr ({INTER_VARIANT}) ({label}) {name}",
            xlabel=name.split("_vs_")[0], ylabel=name.split("_vs_")[1],
            out_png=os.path.join(CORR_OUT, f"region_timevar_scatter_{label}_{name}.png")
        )

    _plot_flat(rtv_l1, rtv_l2, "layer1_vs_layer2")
    _plot_flat(rtv_l1, rtv_sum, "layer1_vs_sum")
    _plot_flat(rtv_l2, rtv_sum, "layer2_vs_sum")

    heatmap_corr_matrix(
        pearson_vals,
        f"REGION TIME-VAR Corr ({INTER_VARIANT}) ({label})",
        os.path.join(CORR_OUT, f"region_timevar_corr_heatmap_{label}.png")
    )

print("DONE: correlations saved to:", CORR_OUT)